In [1]:
# import google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
# load an example merged netcdf file
merged_monthly_file = xr.open_dataset('/content/drive/MyDrive/capstone_data/netCDF/eastern_east_africa_CanESM5_merged.nc')
merged_monthly_file_df = merged_monthly_file.to_dataframe().reset_index().dropna()

In [35]:
# function to create the season category general
short_shifts = [0.5, 1.5]
medium_shifts = [2.5, 3.5]
long_shifts = [4.5, 5.5, 6.5]
def make_season_dict(months, season_prefix, short_shifts, medium_shifts, long_shifts):
    short = [months[0] - s for s in short_shifts]
    medium = [months[0] - s for s in medium_shifts]
    long = [months[0] - s for s in long_shifts]
    return {
        f'{season_prefix}_short': short,
        f'{season_prefix}_medium': medium,
        f'{season_prefix}_long': long,
        'months': months
    }

In [36]:
# use this function when Dec is the start
short_shifts_DJF = [0.5, 1.5, 12.5, 13.5]
medium_shifts_DJF = [2.5, 3.5, 14.5, 15.5]
long_shifts_DJF = [4.5, 5.5, 6.5, 16.5, 17.5, 18.5]
def make_season_dict_DJF(months, season_prefix, short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF):
    short = [months[0] - s for s in short_shifts_DJF]
    medium = [months[0] - s for s in medium_shifts_DJF]
    long = [months[0] - s for s in long_shifts_DJF]
    return {
        f'{season_prefix}_short': short,
        f'{season_prefix}_medium': medium,
        f'{season_prefix}_long': long,
        'months': months
    }

In [37]:
regions_seasons_dict = {
    'eastern_east_africa': {
        'OND': make_season_dict([10, 11, 12], 'OND', short_shifts, medium_shifts, long_shifts),
        'MAM': make_season_dict([3, 4, 5], 'MAM', short_shifts, medium_shifts, long_shifts)
    },
    'lake_victoria': {
        'DJF': make_season_dict_DJF([12, 1, 2], 'DJF', short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF),
        'MAM': make_season_dict([3, 4, 5], 'MAM', short_shifts, medium_shifts, long_shifts),
        'SON': make_season_dict([9, 10, 11], 'SON', short_shifts, medium_shifts, long_shifts)
    },
    'west_africa': {
        'JAS': make_season_dict([7, 8, 9], 'JAS', short_shifts, medium_shifts, long_shifts)
    },
    'southern_africa': {
        'DJF': make_season_dict_DJF([12, 1, 2], 'DJF', short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF),
        'FMA': make_season_dict([2, 3, 4], 'FMA', short_shifts, medium_shifts, long_shifts)
    },
    'south_sudan': {
        'MJJ':make_season_dict([5, 6, 7], 'MJJ', short_shifts, medium_shifts, long_shifts),
        'JAS':make_season_dict([7, 8, 9], 'JAS', short_shifts, medium_shifts, long_shifts),
        'ASO':make_season_dict([8, 9, 10], 'ASO', short_shifts, medium_shifts, long_shifts)
    },
    'eastern_ukraine': {
        'DJF':make_season_dict_DJF([12, 1, 2], 'DJF', short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF),
        'AMJ':make_season_dict([4, 5, 6], 'AMJ', short_shifts, medium_shifts, long_shifts),
        'JA':make_season_dict([7, 8], 'JA', short_shifts, medium_shifts, long_shifts),
    },
    'sri_lanka': {
        'OND':make_season_dict([10, 11, 12], 'OND', short_shifts, medium_shifts, long_shifts)
    }
}



In [38]:
regions_seasons_dict

{'eastern_east_africa': {'OND': {'OND_short': [9.5, 8.5],
   'OND_medium': [7.5, 6.5],
   'OND_long': [5.5, 4.5, 3.5],
   'months': [10, 11, 12]},
  'MAM': {'MAM_short': [2.5, 1.5],
   'MAM_medium': [0.5, -0.5],
   'MAM_long': [-1.5, -2.5, -3.5],
   'months': [3, 4, 5]}},
 'lake_victoria': {'DJF': {'DJF_short': [11.5, 10.5, -0.5, -1.5],
   'DJF_medium': [9.5, 8.5, -2.5, -3.5],
   'DJF_long': [7.5, 6.5, 5.5, -4.5, -5.5, -6.5],
   'months': [12, 1, 2]},
  'MAM': {'MAM_short': [2.5, 1.5],
   'MAM_medium': [0.5, -0.5],
   'MAM_long': [-1.5, -2.5, -3.5],
   'months': [3, 4, 5]},
  'SON': {'SON_short': [8.5, 7.5],
   'SON_medium': [6.5, 5.5],
   'SON_long': [4.5, 3.5, 2.5],
   'months': [9, 10, 11]}},
 'west_africa': {'JAS': {'JAS_short': [6.5, 5.5],
   'JAS_medium': [4.5, 3.5],
   'JAS_long': [2.5, 1.5, 0.5],
   'months': [7, 8, 9]}},
 'southern_africa': {'DJF': {'DJF_short': [11.5, 10.5, -0.5, -1.5],
   'DJF_medium': [9.5, 8.5, -2.5, -3.5],
   'DJF_long': [7.5, 6.5, 5.5, -4.5, -5.5, -6.5],

In [ ]:
def convert_monthly_to_seasonal(file_path, regions_seasons_dict, save_path):
  """
  This function takes a merged monthly netcdf file and converts it to seasonal

  Arguments:
  - file_path: path to merged monthly netcdf file
  - regions_seasons_dict: dictionary of regions and their seasons
  - save_path: path to save the seasonal netcdf file

  Usage Notes:
  Ensure that the file_path follows this format:
  '/content/drive/MyDrive/data/netCDF/eastern_east_africa_CanESM5_merged.nc'
  It does not matter what the netcdf file name is, as long as it follows the format of
  some_region_here_model_merged.nc

  Ensure that the save_path follows this format:
  '/content/drive/MyDrive/data/netCDF'
  Again, it does not matter what the folder name is, as long as it does not end with a '/' or anything else after the folder name.

  Data Notes:
  The merged monthly netcdf file used have the following columns:
  - latitude
  - longitude
  - predicted_precip
  - actual_precip (from CHIRPS)
  - date
  - lead_time

  This merged data has been pre-processed with the monthly_merged_data_generation.py script.

  Regions and Seasons Notes:
  The regions_seasons_dict is a dictionary of regions, their seasons, and specific values of month minus lead time.
  Please refer to the above matrix of month minus lead time values to understand how these values were determined,
  as well as how the dictionary works.
  """

  # extract relevant information from the file path
  split = file_path.split('/') # split into list

  file_name = split[-1] # get the file name

  name_split = file_name.split('_') # get the name of the region, i.e [eastern, east, africa]

  region_name = '_'.join(name_split[0:-2]) # combine the name of the region, i.e eastern_east_africa

  new_file_name= file_name.replace('.nc', '_seasonal.nc') # make new file name for saving

  # check if region name is in regions_seasons_dict
  if region_name not in regions_seasons_dict:
      print(f"ValueError: Region '{region_name}' not found in regions_seasons_dict.")
      return

  # open file
  merged_monthly_file = xr.open_dataset(file_path)

  # convert to a dataframe for pre-processing
  merged_monthly_file_df = merged_monthly_file.to_dataframe().reset_index().dropna()

  # seperate month and year into seperate columns
  merged_monthly_file_df['month'] = merged_monthly_file_df['time'].dt.month
  merged_monthly_file_df['year'] = merged_monthly_file_df['time'].dt.year

  # create month_minus_lead_time
  merged_monthly_file_df['month_minus_lead_time'] = merged_monthly_file_df['month'] - merged_monthly_file_df['lead_time']

  # access the dictionary items of the given region
  region_name = 'eastern_east_africa'
  season_data = regions_seasons_dict[region_name]

  # keep track of the number of seasons in the region
  i = 0

  # Iterate through seasons in the region
  for season_name, season_dict in season_data.items():
      months = season_dict['months']

      # Filter only the rows for the relevant season months, i.e only OND months
      seasonal_df = merged_monthly_file_df[merged_monthly_file_df['month'].isin(months)].copy()

      # Create a new column for the current season
      i += 1
      column_name = f"season_label_{i}"
      merged_monthly_file_df[column_name] = None  # initialize empty season column

      # Iterate over each lead category (i.e OND_short)
      for lead_label, lead_values in season_dict.items():
          if lead_label != 'months':
              # Find matching rows based on month_minus_lead_time
              matching_idx = seasonal_df[seasonal_df['month_minus_lead_time'].isin(lead_values)].index

              # Assign the lead_label to the season column in original dataframe
              merged_monthly_file_df.loc[matching_idx, column_name] = lead_label

  # drop uneccesary columns
  merged_monthly_file_df.drop(['month', 'year', 'month_minus_lead_time'], axis=1, inplace=True)

  return merged_monthly_file_df

  # take this dataframe, convert to netcdf, and store in path
  # seasonal_data_xarray = merged_monthly_file_df.set_index(['time', 'latitude', 'longitude']).to_xarray()

  # save to NetCDF
  # seasonal_data_xarray.to_netcdf(f"{save_path}/{new_file_name}")

In [ ]:
file_path = '/content/drive/MyDrive/capstone_data/netCDF/eastern_east_africa_CanESM5_merged.nc'
save_path = '/content/drive/MyDrive/capstone_data/netCDF/Seasonal'
eea_seasonal_data = convert_monthly_to_seasonal(file_path, regions_seasons_dict, save_path)

In [ ]:
eea_seasonal_data[eea_seasonal_data['season_label_2'] == 'MAM_short']